In [1]:
from pathlib import Path

import polars as pl

DATA_DIR = Path("../data/raw/hf/monthly_processed_data")
sorted(p.name for p in DATA_DIR.glob("*.parquet"))

MONTH = "2024-07"  # change me: any downloaded month

In [2]:
stations = pl.read_csv("../config/stations.csv", schema_overrides={"eva": pl.String})
groups = pl.read_csv("../config/train_type_groups.csv").select("train_type", "group")

print(stations.height, "EVA numbers,", stations["collect"].sum(), "with StaDa data")
groups.group_by("group").len()

146 EVA numbers, 134 with StaDa data


group,len
str,u32
"""long_distance""",18
"""s_bahn""",9
"""regional""",43
"""exclude""",2


## 2. Rows and target

Rules from the README (section *Target and features*). The counts show how many rows each rule removes.

In [3]:
arrivals_raw = (
    pl.scan_parquet(DATA_DIR / f"data-{MONTH}.parquet")
    .select(
        "id", "eva", "train_type", "line_number", "final_destination_station", "train_line_station_num",
        "has_plan_record", "arrival_planned_time", "arrival_change_time", "arrival_is_canceled"
    )
    .filter(pl.col("eva").is_in(stations["eva"].implode()))
    .collect()
    .join(groups, on="train_type", how="left")  # adds the column "group"
)

rules = {
    "rail EVA number (collect = true)": pl.col("eva").is_in(stations.filter(pl.col("collect"))["eva"].implode()),
    "train group is long_distance / regional / s_bahn": pl.col("group").is_in(["long_distance", "regional", "s_bahn"]),
    "has a planned arrival (not the first stop)": pl.col("arrival_planned_time").is_not_null(),
    "arrival not cancelled": ~pl.col("arrival_is_canceled"),
    "stop is in the timetable (has_plan_record)": pl.col("has_plan_record"),
}

arrivals = arrivals_raw
print(f"{'all rows at our 146 EVA numbers':52} {arrivals.height:>10,}")
for name, rule in rules.items():
    arrivals = arrivals.filter(rule)
    print(f"{'+ ' + name:52} {arrivals.height:>10,}")

all rows at our 146 EVA numbers                       2,029,509
+ rail EVA number (collect = true)                    1,981,930
+ train group is long_distance / regional / s_bahn    1,908,261
+ has a planned arrival (not the first stop)          1,489,182
+ arrival not cancelled                               1,418,566
+ stop is in the timetable (has_plan_record)          1,405,544


In [4]:
arrivals = (
    arrivals.join(stations.select("eva", "stada_name", "federal_state"), on="eva", how="left")
    .rename({"group": "train_group"})
    .with_columns(arrival_delay_min=(pl.col("arrival_change_time") - pl.col("arrival_planned_time")).dt.total_minutes())
    .with_columns(y=(pl.col("arrival_delay_min") >= 6).cast(pl.Int8))
)
print(f"Share of late arrivals (y = 1): {arrivals['y'].mean():.1%}")
arrivals.head()

Share of late arrivals (y = 1): 20.7%


id,eva,train_type,line_number,final_destination_station,train_line_station_num,has_plan_record,arrival_planned_time,arrival_change_time,arrival_is_canceled,train_group,stada_name,federal_state,arrival_delay_min,y
str,str,str,str,str,i32,bool,datetime[ns],datetime[ns],bool,str,str,str,i64,i8
"""2444314634072177884-2406302048…","""08000036""","""RE""","""6""","""Bielefeld Hbf""",23,true,2024-07-01 00:01:00,2024-07-01 00:00:00,false,"""regional""","""Bielefeld Hbf""","""Nordrhein-Westfalen""",-1,0
"""1695252692593883835-2406302341…","""08098549""","""S""","""5""","""Hamburg Elbgaustraße""",7,true,2024-06-30 23:59:00,2024-06-30 23:59:00,false,"""s_bahn""","""Hamburg Hbf""","""Hamburg""",0,0
"""-2271920085331621501-240630231…","""08010085""","""S""","""1""","""Meißen Triebischtal""",15,true,2024-06-30 23:58:00,2024-07-01 00:00:00,false,"""s_bahn""","""Dresden Hbf""","""Sachsen""",2,0
"""8357206333431226217-2406302323…","""08000073""","""SBB""","""S6""","""Engen""",12,true,2024-06-30 23:56:00,2024-06-30 23:56:00,false,"""s_bahn""","""Singen (Hohentwiel)""","""Baden-Württemberg""",0,0
"""7866289327576932582-2406302301…","""08000085""","""S""","""6""","""Düsseldorf-Rath Mitte""",21,true,2024-06-30 23:58:00,2024-06-30 23:58:00,false,"""s_bahn""","""Düsseldorf Hbf""","""Nordrhein-Westfalen""",0,0


## 3. Time features: hour, weekday, month

Computed from `arrival_planned_time`, not from `time` or `arrival_change_time`: the actual time is not known the day before.

Careful: Python's `datetime.weekday()` counts **0 = Monday … 6 = Sunday**, Polars' `.dt.weekday()` counts **1 = Monday … 7 = Sunday** (ISO). The model does not care, but the API must compute it exactly the same way as the training, and you must know it when reading results.

In [5]:
arrivals_raw.head()

id,eva,train_type,line_number,final_destination_station,train_line_station_num,has_plan_record,arrival_planned_time,arrival_change_time,arrival_is_canceled,group
str,str,str,str,str,i32,bool,datetime[ns],datetime[ns],bool,str
"""-6129702905591104469-240701000…","""0940370""","""Bus""","""S6 (S""","""Bahnhof, Bad Wildbad""",1,true,null,null,false,"""exclude"""
"""-788955727001224364-2406302347…","""0836075""","""STB""","""1""","""Riegelsberg Süd, Riegelsberg""",9,true,2024-07-01 00:00:00,2024-07-01 00:00:00,false,"""regional"""
"""2444314634072177884-2406302048…","""08000036""","""RE""","""6""","""Bielefeld Hbf""",23,true,2024-07-01 00:01:00,2024-07-01 00:00:00,false,"""regional"""
"""1695252692593883835-2406302341…","""08098549""","""S""","""5""","""Hamburg Elbgaustraße""",7,true,2024-06-30 23:59:00,2024-06-30 23:59:00,false,"""s_bahn"""
"""-2271920085331621501-240630231…","""08010085""","""S""","""1""","""Meißen Triebischtal""",15,true,2024-06-30 23:58:00,2024-07-01 00:00:00,false,"""s_bahn"""


In [ ]:
# Your first check, kept: Python's datetime for one single row
row_number = 1
first = arrivals_raw["arrival_planned_time"][row_number]
print(first, "| Python weekday():", first.weekday(), "(0 = Monday)")

# The feature itself: new columns for ALL rows of `arrivals` (after the five rules) at once
arrivals = arrivals.with_columns(
    hour=pl.col("arrival_planned_time").dt.hour(),        # 0-23, local German time
    weekday=pl.col("arrival_planned_time").dt.weekday(),  # 1 = Monday, 7 = Sunday (ISO)
    month=pl.col("arrival_planned_time").dt.month(),      # 1-12
)

# Proof for the weekday numbering: 1 July 2024 was a Monday
print("Polars weekday of 2024-07-01:", pl.select(pl.date(2024, 7, 1).dt.weekday()).item(), "(1 = Monday)")
arrivals.select("arrival_planned_time", "hour", "weekday", "month").head()

## 4. is_holiday

`add_is_holiday` (in `src/db_delay_mlops/features/holidays.py`) only needs **a time column and a station column**. In the background it reads `config/stations.csv`, finds the federal state of each station and checks with the `holidays` package whether that date is a public holiday there.

Pass the **planned** time (`arrival_planned_time`), never `time` or `arrival_change_time`: the actual time is not known the day before.

July has no public holiday in any state, so for `2024-07` all values are `False`. Try `2024-10` (3 October in all states, 31 October only in some) or `2025-05`.

In [11]:
from datetime import datetime

from db_delay_mlops.features.holidays import add_is_holiday, is_holiday, public_holidays

# For a whole DataFrame: time column + station column in, is_holiday out
arrivals = add_is_holiday(arrivals, time_col="arrival_planned_time", station_col="stada_name")
print(arrivals["is_holiday"].value_counts())

# For a single stop, e.g. later in the API or the Streamlit app
print("31 Oct 2024 in Hamburg Hbf:", is_holiday(datetime(2024, 10, 31, 8, 0), "Hamburg Hbf"))  # Reformationstag
print("31 Oct 2024 in Köln Hbf:   ", is_holiday(datetime(2024, 10, 31, 8, 0), "Köln Hbf"))  # not in NRW

# Which public holidays fall into this month, and in which states?
(
    public_holidays([int(MONTH[:4])])
    .filter(pl.col("date").dt.strftime("%Y-%m") == MONTH)
    .group_by("date", "holiday_name")
    .agg(pl.col("federal_state").sort())
    .sort("date")
)

DuplicateError: column with name 'is_holiday_right' already exists

You may want to try:
- renaming the column prior to joining
- using the `suffix` parameter to specify a suffix different to the default one ('_right')

## 5. Weather: forecast made the day before

`add_weather` (in `src/db_delay_mlops/features/weather.py`) also only needs **a time column and a station column**. In the background it

1. reads the coordinates of each station from `config/stations.csv`,
2. finds all months that occur in the time column,
3. loads each month from `data/raw/weather/weather-YYYY-MM.parquet`, or asks the Open-Meteo Previous Runs API for it first and saves it (5 requests with 25 stations each, a few seconds per month),
4. joins the forecast for the station and the hour of the planned arrival.

The values are the forecast made **24 hours earlier** (`_previous_day1`), not the measured weather.

| WMO code | Meaning |
| --- | --- |
| 0 | clear sky |
| 1–3 | partly cloudy to overcast |
| 45, 48 | fog |
| 51–57 | drizzle |
| 61–67 | rain (65 = heavy) |
| 71–77 | snowfall |
| 80–82 | rain showers (82 = violent) |
| 85, 86 | snow showers |
| 95–99 | thunderstorm |

In [12]:
from db_delay_mlops.features.weather import add_weather, weather_at

# For a whole DataFrame: time column + station column in, weather columns out
arrivals = add_weather(arrivals, time_col="arrival_planned_time", station_col="stada_name")
print("rows without weather:", arrivals["temperature_2m"].null_count())

# For a single stop
print("Köln Hbf, 15 Jul 2024 17:42:", weather_at(datetime(2024, 7, 15, 17, 42), "Köln Hbf"))

arrivals.select("stada_name", "arrival_planned_time", "temperature_2m", "precipitation", "weather_code", "y").head()

rows without weather: 0
Köln Hbf, 15 Jul 2024 17:42: {'temperature_2m': 27.9, 'precipitation': 0.1, 'weather_code': 3}


stada_name,arrival_planned_time,temperature_2m,precipitation,weather_code,y
str,datetime[ns],f64,f64,i16,i8
"""Bielefeld Hbf""",2024-07-01 00:01:00,15.8,0.0,3,0
"""Hamburg Hbf""",2024-06-30 23:59:00,15.8,0.0,3,0
"""Dresden Hbf""",2024-06-30 23:58:00,18.3,0.2,61,0
"""Singen (Hohentwiel)""",2024-06-30 23:56:00,15.4,0.0,3,0
"""Düsseldorf Hbf""",2024-06-30 23:58:00,16.9,0.5,3,0


In [13]:
# Late share per weather code: is the train later when the forecast said rain or thunderstorm?
arrivals.group_by("weather_code").agg(
    pl.len().alias("arrivals"),
    pl.col("y").mean().round(3).alias("late_share"),
).sort("weather_code")

weather_code,arrivals,late_share
i16,u32,f64
0,147193,0.191
1,152674,0.199
2,261541,0.2
3,643392,0.209
45,4649,0.137
…,…,…
80,49282,0.228
81,1138,0.193
95,11764,0.23


### Summary: late arrivals per weather category

The 16 WMO codes are grouped into 8 readable categories with `weather_category()` from `weather.py`. Remember: this is the **forecast** made the day before, not the weather that actually happened.

In [ ]:
from db_delay_mlops.features.weather import weather_category

overall = arrivals["y"].mean()
summary = (
    arrivals.with_columns(weather=weather_category("weather_code"))
    .group_by("weather")
    .agg(
        arrivals=pl.len(),
        late=pl.col("y").sum(),
        late_share=pl.col("y").mean(),
    )
    .with_columns(
        share_of_arrivals=pl.col("arrivals") / pl.col("arrivals").sum(),
        vs_overall=pl.col("late_share") - overall,  # + = later than the month's average
    )
    .sort("late_share", descending=True)
)
print(f"{MONTH}: {arrivals.height:,} arrivals, {arrivals['y'].sum():,} late ({overall:.1%})")
with pl.Config(tbl_rows=20, float_precision=3):
    display(summary)

In [ ]:
# Same idea for temperature: heat has no WMO code, so we look at temperature bands.
# bin_intervals: breakpoints [0, 10, 20, 25, 30] give 6 bins, each needs a label; bins are [left, right)
(
    arrivals.with_columns(
        temperature=pl.col("temperature_2m").bin_intervals(
            [0, 10, 20, 25, 30], labels=["< 0 °C", "0-10 °C", "10-20 °C", "20-25 °C", "25-30 °C", ">= 30 °C"]
        )
    )
    .group_by("temperature")
    .agg(arrivals=pl.len(), late=pl.col("y").sum(), late_share=pl.col("y").mean().round(3))
    .sort("temperature")
)

## Questions

1. Which rule removes the most rows? Is that what you expected?
2. Does the late share change on holidays? (`arrivals.group_by("is_holiday").agg(pl.col("y").mean())`)
3. Is the train later when more precipitation was forecast? Try bins: `pl.col("precipitation").bin_intervals([0.1, 1, 5], labels=["dry", "light", "moderate", "heavy"])`
4. We only use the weather **at the station of arrival**. What does the model miss for a long ICE ride?